# Silver — Central Bank Indicators
Extracts policy rate and CPI from Bronze, fixes date formats, and maps metrics to clean business names.

**Source:** `bronze.central_bank_raw`  
**Output:** `silver.central_bank_indicators` (Delta table)  
**Metrics:** `policy_rate`, `cpi`

In [ ]:
df_raw = spark.read.table("bronze.central_bank_raw")
df_raw.createOrReplaceTempView("v_cb_raw")

In [ ]:
df_silver = spark.sql("""
    WITH raw_parsed AS (
        SELECT
            TO_DATE(date_raw, 'M/d/yyyy h:mm:ss a') AS date,
            CAST(value_raw AS DOUBLE)               AS value,
            series_name,
            ingested_at
        FROM v_cb_raw
    ),
    metric_mapped AS (
        SELECT
            date,
            value,
            CASE 
                WHEN series_name LIKE 'Meginvextir%' THEN 'policy_rate'
                WHEN series_name LIKE 'Vísitala neysluverðs%' THEN 'cpi'
                ELSE NULL 
            END AS metric,
            ingested_at
        FROM raw_parsed
        WHERE date IS NOT NULL
    ),
    deduplicated AS (
        SELECT
            date,
            value,
            metric,
            ingested_at,
            ROW_NUMBER() OVER (PARTITION BY date, metric ORDER BY ingested_at DESC) as rn
        FROM metric_mapped
        WHERE metric IS NOT NULL AND value IS NOT NULL
    )
    SELECT 
        date, 
        metric, 
        value, 
        CURRENT_TIMESTAMP() AS processed_at
    FROM deduplicated
    WHERE rn = 1
""")

df_silver.createOrReplaceTempView("v_cb_silver_staged")

if df_silver.isEmpty():
    mssparkutils.notebook.exit("no-data")

In [ ]:
if not spark.catalog.tableExists("silver.central_bank_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.central_bank_indicators")
    print("Created silver.central_bank_indicators.")
else:
    spark.sql("""
        MERGE INTO silver.central_bank_indicators AS target
        USING v_cb_silver_staged AS source
        ON target.date = source.date
        AND target.metric = source.metric
        WHEN MATCHED THEN UPDATE SET
            target.value = source.value,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (date, metric, value, processed_at)
        VALUES (source.date, source.metric, source.value, source.processed_at)
    """)

print(f"Merge complete. Final count: {spark.table('silver.central_bank_indicators').count()}")